# EV3 Notebook 1: Limpieza e Imputacion de Datos Faltantes
## Bank Marketing Dataset

Identificacion y tratamiento de los valores faltantes enmascarados como `unknown` en el dataset.

**Siguiente:** `EV3_02_Escalamiento_Codificacion.ipynb`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
os.makedirs('graficos', exist_ok=True)

# Cargar dataset
df = pd.read_csv('bank-additional-full.csv', sep=';')
df = df.drop_duplicates()

# Descartar 'duration' por ser data leaker
if 'duration' in df.columns:
    df = df.drop(columns=['duration'])

# Renombrar columnas al espanol
rename_columns = {
    'y': 'deposito_plazo', 'age': 'edad', 'job': 'trabajo',
    'marital': 'estado_civil', 'education': 'educacion', 'default': 'mora',
    'housing': 'vivienda', 'loan': 'prestamo', 'contact': 'contacto',
    'month': 'mes', 'day_of_week': 'dia_de_la_semana', 'campaign': 'campana',
    'pdays': 'dias_previos', 'previous': 'anterior', 'poutcome': 'resultado_anterior',
    'emp.var.rate': 'var_empleo', 'cons.price.idx': 'indice_precios',
    'cons.conf.idx': 'indice_confianza', 'euribor3m': 'tasa_euribor',
    'nr.employed': 'num_empleados'
}
df = df.rename(columns=rename_columns)
df['deposito_plazo'] = df['deposito_plazo'].map({'yes': 'si', 'no': 'no'})
df['deposito_plazo_num'] = df['deposito_plazo'].map({'si': 1, 'no': 0})
df_ml = df.copy()

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")

## Seccion 1: Tratamiento de Datos Faltantes

En este dataset los valores faltantes no aparecen como NaN, sino como la cadena `unknown`. Esto ocurre en datos de contact centers donde el operador no registra la informacion.

**Tecnica seleccionada: Imputacion por la Moda**

Se usa la moda en lugar de eliminar filas porque:
1. Las filas afectadas representan hasta el 20% del dataset, eliminarlas reduce significativamente el volumen de datos.
2. Para variables nominales como trabajo, educacion o estado_civil no existe escala numerica, por lo que la moda es el estimador mas adecuado.
3. Permite que el pipeline procese todas las filas sin interrupciones.

In [ ]:
# Diagnostico de unknowns por columna
cols_check = ['trabajo', 'estado_civil', 'educacion', 'mora', 'vivienda', 'prestamo']
desconocidos = {}
for col in cols_check:
    n = (df_ml[col] == 'unknown').sum()
    pct = round(n / len(df_ml) * 100, 2)
    desconocidos[col] = {'Cantidad': n, 'Porcentaje (%)': pct}

tabla_faltantes = pd.DataFrame(desconocidos).T
tabla_faltantes = tabla_faltantes[tabla_faltantes['Cantidad'] > 0]
print(tabla_faltantes.to_string())

In [ ]:
# Grafico: Porcentaje de unknowns por variable
fig, ax = plt.subplots(figsize=(10, 5))
colores = sns.color_palette("Reds_r", len(tabla_faltantes))
bars = ax.barh(tabla_faltantes.index, tabla_faltantes['Porcentaje (%)'],
               color=colores, edgecolor='white')
for bar, (_, row) in zip(bars, tabla_faltantes.iterrows()):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f"{row['Porcentaje (%)']:.1f}% ({int(row['Cantidad'])} filas)",
            va='center', fontsize=10)
ax.set_xlabel('Porcentaje de unknowns (%)')
ax.set_title('Diagnostico de Datos Faltantes: Unknowns por Variable')
ax.set_xlim(0, tabla_faltantes['Porcentaje (%)'].max() + 10)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig('graficos/datos_faltantes_diagnostico_unknowns.png', bbox_inches='tight')
plt.show()

In [ ]:
# Imputacion por la moda en columnas con unknown
for col in cols_check:
    if 'unknown' in df_ml[col].values:
        moda = df_ml[df_ml[col] != 'unknown'][col].mode()[0]
        n_imp = (df_ml[col] == 'unknown').sum()
        df_ml[col] = df_ml[col].replace('unknown', moda)
        print(f"{col}: {n_imp} valores imputados con moda = '{moda}'")

# Verificacion post-imputacion
print()
print("Verificacion (deben ser 0):")
print({col: (df_ml[col] == 'unknown').sum() for col in cols_check})